# Exploratory Data Analysis — Carbon Intensity & Renewable Fraction
## Thesis: Carbon-Intelligent Workload Scheduling for AI Data Centers
**Author**: Yaxin (Isabel) Wu | **Supervisor**: Prof. Bissan Ghaddar | IE University, 2026

---

### Overview
This notebook performs EDA on hourly Carbon Intensity (CI) and Renewable Fraction (RF) data retrieved from the ElectricityMaps API for five target grid regions: **PJM, NYISO, Finland, Belgium, and Singapore**, covering **2024–2025** (17,521 hours per region).

The analysis proceeds in the following steps:
1. Data loading and basic inspection
2. Descriptive statistics
3. CI time series and distribution
4. CI Variability (CV) — the key regional scheduling opportunity metric
5. Diurnal and seasonal patterns
6. Renewable Fraction (RF) analysis
7. CI–RF relationship
8. Summary of findings and implications for the optimization model

---
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

# Region metadata
ZONES = {
    "US-MIDA-PJM": {"label": "PJM",       "color": "#e05c5c"},
    "US-NY-NYIS":  {"label": "NYISO",     "color": "#e08c3a"},
    "FI":          {"label": "Finland",   "color": "#5ca85c"},
    "BE":          {"label": "Belgium",   "color": "#5c7ae0"},
    "SG":          {"label": "Singapore", "color": "#9b59b6"},
}

print("Imports OK")

---
## 2. Data Loading

We load the pre-fetched `.parquet` files for each zone and merge CI and RF into a single DataFrame per region. The `datetime` column is parsed to UTC-aware timestamps and used as the index throughout the analysis.

In [ ]:
import os

data = {}
for zone, meta in ZONES.items():
    ci = pd.read_parquet(f"../data/raw/{zone}_ci.parquet")
    rf = pd.read_parquet(f"../data/raw/{zone}_rf.parquet")

    df = pd.merge(ci, rf, on="datetime", how="inner")
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
    df = df.set_index("datetime").sort_index()
    df.columns = ["ci", "rf"]
    df["zone"] = meta["label"]
    data[zone] = df

# Quick sanity check
for zone, df in data.items():
    print(f"{ZONES[zone]['label']:10}  rows={len(df):,}  CI_null={df['ci'].isna().sum()}  "
          f"RF_null={df['rf'].isna().sum()}  "
          f"range={df.index.min().date()} → {df.index.max().date()}")

---
## 3. Descriptive Statistics

We compute per-region summary statistics for CI (gCO₂eq/kWh) and RF (%). These numbers directly inform the optimization model: a high CI mean signals a carbon-intensive grid; a high CI std signals strong temporal shifting potential.

In [ ]:
rows = []
for zone, df in data.items():
    rows.append({
        "Region":   ZONES[zone]["label"],
        "CI mean":  round(df["ci"].mean(), 1),
        "CI std":   round(df["ci"].std(), 1),
        "CI min":   int(df["ci"].min()),
        "CI max":   int(df["ci"].max()),
        "CI p25":   int(df["ci"].quantile(0.25)),
        "CI p75":   int(df["ci"].quantile(0.75)),
        "RF mean %": round(df["rf"].mean() * 100, 1),
        "RF std %":  round(df["rf"].std() * 100, 1),
    })

stats = pd.DataFrame(rows).set_index("Region")
stats

---
## 4. CI Variability (CV) — Scheduling Opportunity Metric

**CV (Coefficient of Variation)** = σ(CI) / μ(CI) measures how much the carbon intensity fluctuates relative to its mean. A high CV means large swings between clean and dirty hours, creating more room for temporal shifting to reduce carbon. Singapore's near-zero CV confirms it offers minimal benefit from temporal optimization.

In [ ]:
cv_data = {ZONES[z]["label"]: round(data[z]["ci"].std() / data[z]["ci"].mean(), 4) for z in ZONES}
cv_df = pd.Series(cv_data, name="CV").sort_values(ascending=False).to_frame()
cv_df["Scheduling opportunity"] = ["Highest", "High", "Medium", "Medium", "Lowest"]
print(cv_df)

fig, ax = plt.subplots(figsize=(8, 4))
colors = [ZONES[z]["color"] for z in sorted(ZONES, key=lambda z: -data[z]["ci"].std()/data[z]["ci"].mean())]
bars = ax.barh(cv_df.index, cv_df["CV"], color=colors, edgecolor="white", height=0.6)
ax.set_xlabel("Coefficient of Variation (σ / μ) of CI")
ax.set_title("CI Variability (CV) by Region — Proxy for Scheduling Opportunity")
ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
ax.set_xlim(0, cv_df["CV"].max() * 1.25)
plt.tight_layout()
plt.savefig("../data/eda_cv.png", bbox_inches="tight")
plt.show()

---
## 5. CI Distribution by Region

Boxplots and violin plots reveal the full distribution shape of CI per region. Wide IQR indicates high intra-region variability (PJM, NYISO). Finland shows a right-skewed distribution driven by seasonal nuclear + hydro variation. Singapore's narrow distribution confirms minimal temporal shifting opportunity.

In [ ]:
all_df = pd.concat([df[["ci", "zone"]] for df in data.values()])
zone_order = [ZONES[z]["label"] for z in ZONES]
palette = {ZONES[z]["label"]: ZONES[z]["color"] for z in ZONES}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
sns.boxplot(data=all_df, x="zone", y="ci", order=zone_order,
            palette=palette, width=0.5, fliersize=1.5, ax=axes[0])
axes[0].set_title("CI Distribution — Boxplot")
axes[0].set_xlabel("Region")
axes[0].set_ylabel("Carbon Intensity (gCO₂eq/kWh)")

# Violin
sns.violinplot(data=all_df, x="zone", y="ci", order=zone_order,
               palette=palette, inner="quartile", cut=0, ax=axes[1])
axes[1].set_title("CI Distribution — Violin")
axes[1].set_xlabel("Region")
axes[1].set_ylabel("Carbon Intensity (gCO₂eq/kWh)")

plt.suptitle("Carbon Intensity Distribution by Region (2024–2025, Hourly)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("../data/eda_ci_distribution.png", bbox_inches="tight")
plt.show()

---
## 6. CI Time Series (Monthly Averages)

To visualize long-run trends, we plot monthly-averaged CI for each region. This reveals seasonal structure: Finland's CI drops sharply in summer (hydro + wind peak); PJM shows mild summer peaks from air-conditioning demand driving more gas generation. These seasonal patterns motivate the need for a full 2-year dataset rather than a single snapshot.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for zone, df in data.items():
    monthly = df["ci"].resample("ME").mean()
    ax.plot(monthly.index, monthly.values,
            label=ZONES[zone]["label"],
            color=ZONES[zone]["color"],
            linewidth=2, marker="o", markersize=4)

ax.set_title("Monthly Average Carbon Intensity by Region (2024–2025)")
ax.set_xlabel("Month")
ax.set_ylabel("CI (gCO₂eq/kWh)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig("../data/eda_ci_timeseries.png", bbox_inches="tight")
plt.show()

---
## 7. Diurnal Patterns — Average CI by Hour of Day

Intra-day CI patterns are the primary driver of **temporal shifting** decisions. A large gap between peak-hour and trough-hour CI creates opportunities to defer flexible AI workloads to cleaner windows. We compute the average CI for each hour of the day (UTC) across the full 2-year dataset.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for zone, df in data.items():
    hourly = df["ci"].groupby(df.index.hour).mean()
    diurnal_range = hourly.max() - hourly.min()
    ax.plot(hourly.index, hourly.values,
            label=f"{ZONES[zone]['label']} (range={diurnal_range:.0f})",
            color=ZONES[zone]["color"], linewidth=2, marker="o", markersize=3)

ax.set_title("Average CI by Hour of Day (UTC) — Diurnal Pattern")
ax.set_xlabel("Hour of Day (UTC)")
ax.set_ylabel("Mean CI (gCO₂eq/kWh)")
ax.set_xticks(range(0, 24, 2))
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig("../data/eda_diurnal.png", bbox_inches="tight")
plt.show()

print("\nDiurnal range (max - min avg CI by hour):")
for zone, df in data.items():
    hourly = df["ci"].groupby(df.index.hour).mean()
    print(f"  {ZONES[zone]['label']:10}  {hourly.max() - hourly.min():.1f} gCO₂/kWh")

---
## 8. Seasonal Patterns — Average CI by Month

Seasonal variation drives **long-horizon temporal shifting** and explains why Finland's CV is deceptively moderate despite its very low mean CI. We compute mean CI per calendar month to expose these patterns.

In [ ]:
months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig, ax = plt.subplots(figsize=(12, 5))

for zone, df in data.items():
    monthly = df["ci"].groupby(df.index.month).mean()
    ax.plot(monthly.index, monthly.values,
            label=ZONES[zone]["label"],
            color=ZONES[zone]["color"], linewidth=2, marker="s", markersize=5)

ax.set_title("Average CI by Month — Seasonal Pattern (2024–2025 combined)")
ax.set_xlabel("Month")
ax.set_ylabel("Mean CI (gCO₂eq/kWh)")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(months)
ax.legend()
plt.tight_layout()
plt.savefig("../data/eda_seasonal.png", bbox_inches="tight")
plt.show()

---
## 9. Renewable Fraction (RF) Analysis

RF enters the objective function as the weighting term `(1 − α·RF(t))`, reducing the effective carbon cost in hours with high renewable generation. Here we examine RF distributions and its intra-day pattern to understand when the model will preferentially schedule workloads.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RF distribution
rf_all = pd.concat([df[["rf", "zone"]] for df in data.values()])
sns.boxplot(data=rf_all, x="zone", y="rf", order=zone_order,
            palette=palette, width=0.5, ax=axes[0])
axes[0].set_title("RF Distribution by Region")
axes[0].set_xlabel("Region")
axes[0].set_ylabel("Renewable Fraction")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

# RF diurnal pattern
for zone, df in data.items():
    rf_hourly = df["rf"].groupby(df.index.hour).mean()
    axes[1].plot(rf_hourly.index, rf_hourly.values,
                 label=ZONES[zone]["label"],
                 color=ZONES[zone]["color"], linewidth=2)
axes[1].set_title("Average RF by Hour of Day (UTC)")
axes[1].set_xlabel("Hour of Day (UTC)")
axes[1].set_ylabel("Mean Renewable Fraction")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
axes[1].set_xticks(range(0, 24, 2))
axes[1].legend(fontsize=9)

plt.suptitle("Renewable Fraction Analysis (2024–2025)", fontsize=13)
plt.tight_layout()
plt.savefig("../data/eda_rf.png", bbox_inches="tight")
plt.show()

---
## 10. CI–RF Relationship

We examine whether low CI and high RF co-occur (which would mean the two signals point in the same direction) or diverge (which would justify including RF as an independent term). If CI and RF are negatively correlated, the RF weighting term adds real discriminating power beyond CI alone.

In [ ]:
fig, axes = plt.subplots(1, len(ZONES), figsize=(18, 4), sharey=False)

for ax, (zone, df) in zip(axes, data.items()):
    sample = df.sample(min(3000, len(df)), random_state=42)
    corr = df["ci"].corr(df["rf"])
    ax.scatter(sample["rf"], sample["ci"],
               alpha=0.15, s=8, color=ZONES[zone]["color"])
    ax.set_title(f"{ZONES[zone]['label']}\nr = {corr:.3f}")
    ax.set_xlabel("Renewable Fraction")
    ax.set_ylabel("CI (gCO₂eq/kWh)" if ax == axes[0] else "")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

plt.suptitle("CI vs. RF Scatter by Region (sample n=3,000)", fontsize=13)
plt.tight_layout()
plt.savefig("../data/eda_ci_rf_scatter.png", bbox_inches="tight")
plt.show()

print("\nPearson correlation (CI, RF):")
for zone, df in data.items():
    print(f"  {ZONES[zone]['label']:10}  r = {df['ci'].corr(df['rf']):.3f}")

---
## 11. Key Findings & Implications for the Optimization Model

This section consolidates the EDA findings and maps them directly to modeling decisions.

In [ ]:
summary = []
for zone, df in data.items():
    cv = df["ci"].std() / df["ci"].mean()
    diurnal_range = df["ci"].groupby(df.index.hour).mean()
    ci_rf_corr = df["ci"].corr(df["rf"])
    summary.append({
        "Region":         ZONES[zone]["label"],
        "CI mean":        round(df["ci"].mean(), 1),
        "CV":             round(cv, 3),
        "Diurnal range":  round(diurnal_range.max() - diurnal_range.min(), 1),
        "RF mean %":      round(df["rf"].mean() * 100, 1),
        "CI–RF corr":     round(ci_rf_corr, 3),
        "Scheduling opp": "High" if cv > 0.15 else ("Medium" if cv > 0.08 else "Low"),
    })

summary_df = pd.DataFrame(summary).set_index("Region")
print(summary_df.to_string())
print("""
Key takeaways:
1. PJM and NYISO have the highest CV → largest temporal shifting potential
2. Finland has very low mean CI but moderate CV → seasonal rather than diurnal opportunity
3. Singapore has near-zero CV → minimal benefit from temporal shifting (serves as control case)
4. CI and RF are negatively correlated in all regions (except SG ≈ 0)
   → RF adds discriminating power: low-CI + high-RF hours are genuinely cleaner
   → The RF weighting term in the objective is empirically justified
""")